In [ ]:
import bpy
import bmesh
import matplotlib.pyplot as plt
from mathutils.bvhtree import BVHTree

# --------------------
# CONFIG
# --------------------
COLLECTION_NAME = "Cubes"
PLANE_NAME = "Plane"
COLOR_ATTRIBUTE_NAME = "Col"   # Change if needed
EPSILON = 0.0005

scene = bpy.context.scene
start = scene.frame_start
end = scene.frame_end
depsgraph = bpy.context.evaluated_depsgraph_get()

# --------------------
# Build plane BVH once
# --------------------
plane = bpy.data.objects[PLANE_NAME]
plane_eval = plane.evaluated_get(depsgraph)
plane_mesh = plane_eval.to_mesh()

bm = bmesh.new()
bm.from_mesh(plane_mesh)
bm.transform(plane.matrix_world)
plane_bvh = BVHTree.FromBMesh(bm)
bm.free()

# --------------------
# Access color attribute
# --------------------
color_layer = plane_mesh.color_attributes.get(COLOR_ATTRIBUTE_NAME)

if color_layer is None:
    raise ValueError(f"Color attribute '{COLOR_ATTRIBUTE_NAME}' not found on {PLANE_NAME}")

color_data = color_layer.data

# --------------------
# Get all cubes from the collection
# --------------------
if COLLECTION_NAME not in bpy.data.collections:
    raise ValueError(f"Collection '{COLLECTION_NAME}' not found")

cubes_collection = bpy.data.collections[COLLECTION_NAME]
cubes = [obj for obj in cubes_collection.objects if obj.type == 'MESH']

if not cubes:
    raise ValueError(f"No mesh objects found in collection '{COLLECTION_NAME}'")

print(f"Found {len(cubes)} cubes in collection '{COLLECTION_NAME}'")

# --------------------
# Scan timeline
# --------------------
original_frame = scene.frame_current

frames = []
distances = {cube.name: [] for cube in cubes}
collision_frames = {cube.name: [] for cube in cubes}
collision_colors = {cube.name: [] for cube in cubes}

colors_r = {cube.name: [] for cube in cubes}
colors_g = {cube.name: [] for cube in cubes}
colors_b = {cube.name: [] for cube in cubes}

was_colliding = {cube.name: False for cube in cubes}

for frame in range(start, end + 1):
    scene.frame_set(frame)
    
    if frame == start:
        frames.append(frame)
    elif frame not in frames:
        frames.append(frame)

    for cube in cubes:
        center = cube.location
        # Use average dimension for radius approximation
        radius = max(cube.dimensions) / 2

        loc, normal, face_index, dist = plane_bvh.find_nearest(center)

        if loc is None:
            continue

        surface_distance = dist - radius

        distances[cube.name].append(surface_distance)

        # --------------------
        # Read vertex paint color from closest face
        # --------------------
        poly = plane_mesh.polygons[face_index]

        face_colors = []
        for loop_index in poly.loop_indices:
            c = color_data[loop_index].color
            face_colors.append((c[0], c[1], c[2]))

        if face_colors:
            avg_color = (
                sum(c[0] for c in face_colors) / len(face_colors),
                sum(c[1] for c in face_colors) / len(face_colors),
                sum(c[2] for c in face_colors) / len(face_colors),
            )
        else:
            avg_color = (0.0, 0.0, 0.0)

        colors_r[cube.name].append(avg_color[0])
        colors_g[cube.name].append(avg_color[1])
        colors_b[cube.name].append(avg_color[2])

        # --------------------
        # Bounce detection
        # --------------------
        is_colliding = surface_distance <= EPSILON

        if is_colliding and not was_colliding[cube.name]:
            collision_frames[cube.name].append(frame)
            collision_colors[cube.name].append(avg_color)

            print(
                f"Bounce detected for {cube.name} at frame {frame} | "
                f"RGB = ({avg_color[0]:.3f}, "
                f"{avg_color[1]:.3f}, "
                f"{avg_color[2]:.3f})"
            )

        was_colliding[cube.name] = is_colliding

scene.frame_set(original_frame)
plane_eval.to_mesh_clear()

for cube in cubes:
    print(f"\nBounce frames for {cube.name}:", collision_frames[cube.name])

# --------------------
# Plot distance
# --------------------
plt.figure()

for cube in cubes:
    plt.plot(frames[:len(distances[cube.name])], distances[cube.name], label=cube.name)

plt.axhline(0)

for cube in cubes:
    if collision_frames[cube.name]:
        collision_distances = [distances[cube.name][frames.index(f)] for f in collision_frames[cube.name] if f in frames]
        plt.scatter(collision_frames[cube.name][:len(collision_distances)], collision_distances)

plt.xlabel("Frame")
plt.ylabel("Surface Distance to Plane")
plt.title("Distance to Plane with Collision Events")
plt.legend()

# --------------------
# Plot colors
# --------------------
plt.figure()

for cube in cubes:
    plt.plot(frames[:len(colors_r[cube.name])], colors_r[cube.name], label=f"{cube.name} R")
    plt.plot(frames[:len(colors_g[cube.name])], colors_g[cube.name], label=f"{cube.name} G")
    plt.plot(frames[:len(colors_b[cube.name])], colors_b[cube.name], label=f"{cube.name} B")

plt.xlabel("Frame")
plt.ylabel("Sampled Surface Color")
plt.title("Vertex Paint Color Under Objects (RGB)")
plt.legend()

plt.show()

KeyError: 'bpy_prop_collection[key]: key "Sphere" not found'

23:23.810  operator         | Cleared System Clipboard
